# K-Means 聚类问题 (MSSC)

**类别：** 选址

来源：[https://www.hexaly.com/templates/k-means-clustering-problem-mssc](https://www.hexaly.com/templates/k-means-clustering-problem-mssc)


## 问题

**在 K-Means 聚类问题**（最小平方和聚类，MSSC）中，我们希望将一组多维观测点划分为 k 个聚类，每个聚类由其重心定义。每个观测点属于具有最近重心的聚类。更多细节，请参阅 [Wikipedia](http://en.wikipedia.org/wiki/K-means_clustering)。

	

### 学到的建模原则

- 添加 [set decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模聚类
- 使用 [lambda functions](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算每个聚类的重心和方差


## 数据

数据格式如下：

- 第 1 行：观测点数和每个观测点的维度数
- 对每个观测点：每个维度上的坐标以及其在最优解中所属的聚类


## 模型

K-Means 聚类问题 (MSSC) 的 OptAgent 模型保持 Hexaly 的集合变量建模逻辑，使用 [set decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。每个 set variable 表示一个聚类，其内部的元素表示属于该聚类的观测点。我们使用一个 [**partition** operator](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html#n-ary-operators) 来确保每个观测点恰好属于一个聚类。

我们使用 [lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 对每个聚类内所有观测点在所有维度上的坐标应用 **sum** 算子来计算每个聚类的重心。注意，此求和中项的数量在搜索过程中会随着集合的大小变化而变化。

然后我们可以计算总方差。一个聚类的方差是该聚类重心与每个观测点之间的欧几里得距离平方之和。与重心类似，我们使用 lambda function 计算每个聚类的方差。目标是最小化这些方差之和。


## Results

**Hexaly Optimizer 在 1 分钟运行时间内即可在 K-Means 聚类问题 (MSSC) 上达到低于 1% 的 gap**，这些实例来自 UCI 机器学习仓库和 TSPLIB 研究基准，包含超过 **10,000 个观测点**。我们的 [K-Means Clustering Problem (MSSC) benchmark page](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-k-means-clustering-mssc) 展示了 Hexaly Optimizer 在这一富有挑战性的问题上如何超越 Gurobi 等传统的通用优化求解器。

[Explore this benchmark](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-k-means-clustering-mssc)


## Python 实现


In [1]:
from pathlib import Path

from optagent import OptModel, solve

def read_elem(filename):
    return Path(filename).read_text(encoding="utf-8").split()

#
# Read instance data
#
def read_instance(filename):
    file_it = iter(read_elem(filename))

    # Data properties
    nb_observations = int(next(file_it))
    nb_dimensions = int(next(file_it))

    coordinates_data = [None] * nb_observations
    for o in range(nb_observations):
        coordinates_data[o] = [None] * (nb_dimensions)
        for d in range(nb_dimensions):
            coordinates_data[o][d] = float(next(file_it))
        next(file_it) # skip initial clusters

    return nb_observations, nb_dimensions, coordinates_data

def main(instance_file, output_file=None, time_limit=60, k=2):
    nb_observations, nb_dimensions, coordinates_data = read_instance(instance_file)

    model = OptModel()

    # clusters[c] represents the observations assigned to cluster c.
    clusters = [model.set(nb_observations, name=f"cluster_{c}") for c in range(k)]

    # Each observation must belong to exactly one cluster.
    model.constraint(model.partition(clusters), name="cluster_partition")

    coordinates = model.array(coordinates_data)
    variances = []
    for cluster in clusters:
        size = model.count(cluster)

        # Keep the empty-cluster guard from the Hexaly model.
        centroid = []
        for d in range(nb_dimensions):
            coordinate_lambda = model.lambda_function(
                lambda i: model.at(coordinates, i // 1, d)
            )
            centroid.append(
                model.iif(size == 0, 0.0, model.sum(cluster, coordinate_lambda) / size)
            )

        dimension_variances = []
        for d in range(nb_dimensions):
            centroid_d = centroid[d]
            dimension_variance_lambda = model.lambda_function(
                lambda i: model.pow(
                    model.at(coordinates, i // 1, d) - centroid_d, 2
                )
            )
            dimension_variances.append(model.sum(cluster, dimension_variance_lambda))
        variances.append(model.sum(dimension_variances))

    # Minimize the total within-cluster sum of squares.
    total_variance = model.sum(variances)
    model.minimize(total_variance, name="total_variance")

    solution = solve(model, time_limit_s=float(time_limit))
    print(f"Observations = {nb_observations}; Dimensions = {nb_dimensions}; "
          f"Clusters = {k}; Total variance = {total_variance.value}; "
          f"Status = {solution.feasible}")

    if solution.feasible:
        for c, cluster in enumerate(clusters):
            print(f"Cluster {c}: {sorted(cluster.value)}")
        if output_file is not None:
            lines = [str(total_variance.value), str(k)]
            lines.extend(" ".join(str(o) for o in sorted(cluster.value)) for cluster in clusters)
            Path(output_file).write_text("\n".join(lines) + "\n", encoding="utf-8")
    return solution


## 运行实例

以下代码格演示如何调用 OptAgent 模型。`k` 使用数据集中的类别数；如需更长搜索，可增大 `time_limit`。


In [2]:
INSTANCE_DIR = Path.cwd() / "instances"
print("Instances:", INSTANCE_DIR)


Instances: /Users/dongbox/work/opt-agent/examples/examples/hexaly/k_means_clustering_problem_mssc/instances


In [ ]:
solution_ruspini = main(INSTANCE_DIR / "ruspini.dat", k=4, time_limit=1)


Starting OptAgent
Parameters: time_limit=5s
[   0.001s] initial feasible=false hard_structure_violations=75 violations=1 normalized_violation=1 objective=[0]
[   0.007s] best #1 worker=0 feasible=true objective=[244374]
[   0.288s] best #9 worker=1 feasible=true objective=[220658]
[   0.491s] best #16 worker=1 feasible=true objective=[204168]


Observations = 75; Dimensions = 2; Clusters = 4; Total variance = 193018.1296992481; Status = True
Cluster 0: [19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74]
Cluster 1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Cluster 2: []
Cluster 3: []


[   5.007s] best #20 worker=1 feasible=true objective=[193018]
Solve summary:
  status: FEASIBLE
  objective: [193018]
  improvements: 20
  evaluated: 744
  wall_time: 5.00663s
  termination: deadline


In [ ]:
solution_iris = main(INSTANCE_DIR / "iris.dat", k=3, time_limit=1)


In [ ]:
solution_glass = main(INSTANCE_DIR / "glass.dat", k=6, time_limit=1)


In [ ]:
solution_segment = main(INSTANCE_DIR / "segment.dat", k=7, time_limit=1)
